# 🎯 RAG — Retrieval Augmented Generation

## What is RAG?
**RAG** = give the LLM relevant documents before asking it questions.

Without RAG: LLM uses only its training data (may be outdated or incomplete).
With RAG: LLM uses training data + your specific documents.

## The RAG Pipeline
```
INDEXING (done once):
Documents → Split → Embed → Vector Store

RETRIEVAL (at query time):
Question → Embed → Similarity Search → Top K Docs

GENERATION:
[System Prompt + Retrieved Docs + Question] → LLM → Answer
```

## Why RAG?
| Problem | RAG Solution |
|---------|-------------|
| LLM doesn't know your company docs | Feed docs into context |
| Training data is outdated | Add recent documents |
| LLM hallucinates | Ground answers in real sources |
| Fine-tuning is expensive | RAG is cheap and dynamic |

## RAG Quality Metrics
- **Precision**: Are retrieved docs relevant?
- **Recall**: Are all relevant docs retrieved?
- **Faithfulness**: Does answer match the retrieved docs?
- **Relevance**: Does answer address the question?


In [ ]:
# ── Simple RAG Chain ─────────────────────────────────────────────────────
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── Step 1: Create vector store with knowledge base ──
knowledge_base = [
    Document(page_content='LangChain was released in 2022. Creator: Harrison Chase.'),
    Document(page_content='LangGraph is a library for building stateful multi-actor applications.'),
    Document(page_content='LCEL stands for LangChain Expression Language. Uses | operator.'),
    Document(page_content='Anthropic created Claude AI. OpenAI created ChatGPT.'),
    Document(page_content='RAG stands for Retrieval Augmented Generation.'),
]

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = FAISS.from_documents(knowledge_base, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

# ── Step 2: RAG Prompt ──
rag_prompt = ChatPromptTemplate.from_messages([
    ('system', '''You are an assistant. Answer ONLY using the provided context.
If the context doesn't contain the answer, say "I don't have that information."

Context:
{context}'''),
    ('human', '{question}'),
])

# ── Step 3: Format documents for context ──
def format_docs(docs: list[Document]) -> str:
    return '\n\n'.join(doc.page_content for doc in docs)

# ── Step 4: Build RAG chain ──
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# ── Test it ──
question = 'What does LCEL stand for and what does it use?'
answer = rag_chain.invoke(question)
print(f'Q: {question}')
print(f'A: {answer}')

In [ ]:
# ── RAG with Source Citations ─────────────────────────────────────────
# Production RAG should cite sources so users can verify answers.

from langchain_core.runnables import RunnableParallel
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Docs with source metadata
knowledge_base = [
    Document(page_content='LangChain was created in 2022.', metadata={'source': 'langchain_wiki.pdf', 'page': 1}),
    Document(page_content='LangGraph supports stateful agents.', metadata={'source': 'langgraph_docs.html', 'page': 1}),
]

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = FAISS.from_documents(knowledge_base, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

rag_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Answer based on context. Context:\n{context}'),
    ('human', '{question}'),
])

def format_docs_with_sources(docs):
    formatted = []
    for doc in docs:
        src = doc.metadata.get('source', 'unknown')
        formatted.append(f'[Source: {src}]\n{doc.page_content}')
    return '\n\n'.join(formatted)

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Return BOTH answer and source documents
rag_chain_with_sources = RunnableParallel(
    answer=(
        {'context': retriever | format_docs_with_sources, 'question': RunnablePassthrough()}
        | rag_prompt | llm | StrOutputParser()
    ),
    sources=retriever
)

result = rag_chain_with_sources.invoke('What is LangChain?')
print('Answer:', result['answer'])
print('\nSources used:')
for doc in result['sources']:
    print(f'  - {doc.metadata["source"]}: {doc.page_content[:60]}...')

## 🎯 Interview Questions

1. **What is RAG and why is it used instead of fine-tuning?**
2. **Describe the full RAG pipeline from document to answer.**
3. **What are the 4 quality metrics for RAG systems?**
4. **How do you cite sources in a RAG response?**
5. **What is the role of chunk_overlap in RAG quality?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **RAG — Retrieval Augmented Generation**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
